### Enrollment Module
Enroll user into system
* Create templates based on merged embeddings for each identity

In [4]:
# Build per-person templates from enroll embeddings (mean + L2 normalize), save template_index in map
# - Builds 1 template per person_id 
# - Averages all embeddings for that person_id to create the template
# - Saves templates and templates_map to ../data_processed/vggface2
# - Builds FAISS index (IndexFlatIP) as templates_enroll.index

import numpy as np
import pandas as pd
from pathlib import Path
import faiss
import logging
import random
import math

# Paths
EMB_DIR = Path("../data_processed/vggface2/embeddings/enroll")
OUT_DIR = Path("../data_processed/vggface2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional controls
CAP_PER_PERSON = None    # set to int (e.g. 20) to cap # images used per person, or None to use all
GLOBAL_SEED = 42         # deterministic sampling seed when capping

logger = logging.getLogger("enrollment")
logger.info("Using embeddings dir: %s", EMB_DIR)

# Load merged embeddings + map (merge_shards must have been run)
emb_path = EMB_DIR / "embeddings.npy"
map_path = EMB_DIR / "embeddings_map.csv"
assert emb_path.exists() and map_path.exists(), f"Missing {emb_path} or {map_path}. Run merge_shards first."

emb = np.load(emb_path).astype("float32")
emap = pd.read_csv(map_path)
logger.info("Loaded embeddings: %d rows, map rows: %d", emb.shape[0], len(emap))

# Group by person_id and build templates
groups = emap.groupby("person_id", sort=True).indices
templates = []
tmpl_rows = []

for pid, idxs in groups.items():
    idxs = list(idxs)
    n_total = len(idxs)

    # Cap the number of images used per person (optional)
    if CAP_PER_PERSON is not None and n_total > CAP_PER_PERSON:
        rnd = random.Random((GLOBAL_SEED + (hash(pid) & 0xFFFFFFFF)) & 0xFFFFFFFF)
        rnd.shuffle(idxs)
        selected = idxs[:CAP_PER_PERSON]
    else:
        selected = idxs

    vecs = emb[selected]
    tmpl = vecs.mean(axis=0)
    norm = np.linalg.norm(tmpl)
    if norm == 0:
        logger.warning("Zero-norm template for %s; skipping", pid)
        continue
    tmpl = (tmpl / norm).astype("float32")

    template_index = len(templates)   # global index for this template in the templates array
    templates.append(tmpl)
    # keep small sample of image paths used to create template (pipe-separated, truncated)
    sample_paths = emap.iloc[selected]["image_path"].astype(str).tolist()
    tmpl_rows.append({
        "person_id": pid,
        "n_images": n_total,
        "n_used": len(selected),
        "template_index": template_index,
        "sample_paths": "|".join(sample_paths[:10])
    })

if not templates:
    raise RuntimeError("No templates created (no valid persons found).")

templates = np.stack(templates).astype("float32")

# Save templates and map (clear, explicit names so original templates remain intact)
templates_path = OUT_DIR / "templates_enroll.npy"
templates_map_path = OUT_DIR / "templates_map_enroll.csv"
np.save(templates_path, templates)
pd.DataFrame(tmpl_rows).to_csv(templates_map_path, index=False)
logger.info("Wrote templates: %s rows=%d", templates_path, templates.shape[0])

# Build FAISS index (IndexFlatIP for normalized vectors => inner product ~ cosine)
# Templates are already L2-normalized above, but ensure numeric safety
templates /= np.linalg.norm(templates, axis=1, keepdims=True).clip(min=1e-12)
dim = templates.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(templates)
index_path = OUT_DIR / "templates_enroll.index"
faiss.write_index(index, str(index_path))
logger.info("Built FAISS index: %s (dim=%d, n=%d)", index_path, dim, templates.shape[0])

Wrote templates: ..\data_processed\vggface2\embeddings\enroll\templates_pose.npy rows: 3217
Wrote FAISS index: ..\data_processed\vggface2\embeddings\enroll\templates_pose.index


In [6]:
import numpy as np, pandas as pd
from pathlib import Path

base = Path("../data_processed/vggface2")
t1_npy = base / "templates_enroll.npy"
t1_map = base / "templates_map_enroll.csv"
t2_npy = base / "embeddings/enroll/templates_pose.npy"
t2_map = base / "embeddings/enroll/templates_map_pose.csv"

def info(npy, mapf):
    ok = npy.exists() and mapf.exists()
    print(npy, "exists:", ok)
    if ok:
        tpl = np.load(npy)
        df = pd.read_csv(mapf)
        print(" rows:", len(df), "templates array shape:", tpl.shape)
        print(" types/counts:\n", df.get('type', pd.Series(['pooled']*len(df))).value_counts())
        print(df.head())
    print()

info(t1_npy, t1_map)
info(t2_npy, t2_map)

..\data_processed\vggface2\templates_enroll.npy exists: True
 rows: 540 templates array shape: (540, 512)
 types/counts:
 pooled    540
Name: count, dtype: int64
  person_id  n_images  n_used  template_index  \
0   n000001       339     339               0   
1   n000002       252     252               1   
2   n000003       164     164               2   
3   n000004       309     309               3   
4   n000005       183     183               4   

                                        sample_paths  
0  C:\Users\yukki\Documents\CS228-Biometric-with-...  
1  C:\Users\yukki\Documents\CS228-Biometric-with-...  
2  C:\Users\yukki\Documents\CS228-Biometric-with-...  
3  C:\Users\yukki\Documents\CS228-Biometric-with-...  
4  C:\Users\yukki\Documents\CS228-Biometric-with-...  

..\data_processed\vggface2\embeddings\enroll\templates_pose.npy exists: True
 rows: 3217 templates array shape: (3217, 512)
 types/counts:
 type
per_pose           2677
pooled_fallback     540
Name: count, dtype:

In [7]:
import pandas as pd, numpy as np
from pathlib import Path

base = Path("../data_processed/vggface2")
t1_map = base / "templates_map_enroll.csv"
t2_map = base / "embeddings/enroll/templates_map_pose.csv"

t1 = pd.read_csv(t1_map)
t2 = pd.read_csv(t2_map)

print("baseline persons:", t1['person_id'].nunique(), "rows:", len(t1))
print("pose map persons:", t2['person_id'].nunique(), "rows:", len(t2))
print("pooled_fallback count:", (t2['type']=='pooled_fallback').sum())

# template_idx uniqueness
assert t2['template_idx'].is_unique, "template_idx not unique"
print("template_idx OK; min/max:", t2['template_idx'].min(), t2['template_idx'].max())
# every person has pooled fallback?
pooled_per_person = t2[t2['type']=='pooled_fallback'].groupby('person_id').size()
print("persons missing pooled fallback:", (pooled_per_person==0).sum())

baseline persons: 540 rows: 540
pose map persons: 540 rows: 3217
pooled_fallback count: 540
template_idx OK; min/max: 0 3216
persons missing pooled fallback: 0


In [8]:
# Pose-based enrollment
# Enrollment: build per-person-per-pose (aligned-only) + pooled fallbacks (K=5)
# - Uses embeddings from embeddings_map_with_pose_liveness_k5.csv
# - Saves templates to ../data_processed/vggface2/embeddings/enroll/templates_pose
# - Builds FAISS index for templates as templates_pose.index

# HQ per-pose templates + pooled fallback
import numpy as np
import pandas as pd
from pathlib import Path
import faiss

SPLIT = "enroll"
EMB_DIR = Path(f"../data_processed/vggface2/embeddings/{SPLIT}")
MAP_K5 = EMB_DIR / "embeddings_map_with_pose_liveness_k5.csv"
EMB_NPY = EMB_DIR / "embeddings.npy"
OUT_DIR = EMB_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# === HQ / fallback knobs ===
HQ_MIN_IMAGES_PER_TEMPLATE = 2     # require >= this many HQ images for a per-pose template
HQ_MIN_LIVENESS = 0.50             # min liveness for HQ image
HQ_MIN_SIDE = 80                   # same MIN_FACE_SIDE you used
HQ_MIN_CONF = 0.90                 # optional detector confidence if present
FALLBACK_MIN_LIVENESS = None       # None => pooled uses all images; or set 0.5 to filter
USE_WEIGHTED_MEAN = True           # weight images by quality when averaging

# load
df = pd.read_csv(MAP_K5)
embs = np.load(EMB_NPY).astype(np.float32)

# ensure index mapping
if 'embedding_index' in df.columns:
    emb_idx_array = df['embedding_index'].to_numpy().astype(int)
else:
    assert len(df) == embs.shape[0], "embeddings.npy rows must match map rows"
    emb_idx_array = np.arange(len(df), dtype=int)

# Compute per-image quality score (0..1)
# Components: liveness (0..1), box area -> min_side normalized, optional det_conf (if available)
liveness = df['liveness_score'].fillna(0.5).to_numpy(dtype=float)  # fallback 0.5 neutral
# estimate min side from face_box_area if box_w/box_h not available
min_side_est = np.sqrt(df['face_box_area'].clip(lower=0.0).to_numpy(dtype=float))
# normalize min_side into [0,1] by clamping at e.g. 20..160 px
side_min = 20.0
side_max = 160.0
side_norm = np.clip((min_side_est - side_min) / (side_max - side_min), 0.0, 1.0)
# detector confidence if present
if 'det_conf' in df.columns:
    det_conf = df['det_conf'].fillna(0.0).to_numpy(dtype=float)
else:
    det_conf = None

# combine into a single quality score (weights tuned conservatively)
w_liv, w_side, w_conf = 0.6, 0.3, 0.1
if det_conf is None:
    w_liv, w_side = 0.7, 0.3
    quality = w_liv * liveness + w_side * side_norm
else:
    quality = w_liv * liveness + w_side * side_norm + w_conf * det_conf
# clip to [0,1]
quality = np.clip(quality, 0.0, 1.0)
df['quality_score'] = quality

# Build masks
face_aligned = df['face_aligned'].fillna(False).astype(bool).to_numpy()
hq_mask = face_aligned.copy()
hq_mask &= (liveness >= HQ_MIN_LIVENESS)
hq_mask &= (min_side_est >= HQ_MIN_SIDE)
if det_conf is not None:
    hq_mask &= (det_conf >= HQ_MIN_CONF)

if FALLBACK_MIN_LIVENESS is None:
    fallback_mask = np.ones(len(df), dtype=bool)
else:
    fallback_mask = (liveness >= FALLBACK_MIN_LIVENESS)

# normalized embeddings per-row
def l2norm_rows(x):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return x / n

emb_by_row = embs[emb_idx_array]
emb_by_row = l2norm_rows(emb_by_row)

templates = []
rows = []
tpl_idx = 0

# Helper: weighted / unweighted template builder
def make_template_from_indices(idxs, weights=None):
    vecs = emb_by_row[idxs].astype(np.float32)
    if weights is None:
        tpl = vecs.mean(axis=0)
    else:
        w = np.asarray(weights, dtype=np.float32)
        wsum = w.sum()
        if wsum <= 0:
            tpl = vecs.mean(axis=0)
        else:
            tpl = (vecs * (w[:, None] / (wsum + 1e-12))).sum(axis=0)
    tpl = tpl / (np.linalg.norm(tpl) + 1e-12)
    return tpl

# Build HQ per-pose templates
if 'pose_cluster_k5' not in df.columns:
    raise RuntimeError("Missing 'pose_cluster_k5' in map. Run reclustering to K=5 first.")

grp = df[hq_mask].groupby(['person_id', 'pose_cluster_k5'], sort=True)
for (person_id, pose_k), g in grp:
    if int(pose_k) < 0:
        continue
    idxs = g.index.to_numpy().astype(int)
    if len(idxs) < HQ_MIN_IMAGES_PER_TEMPLATE:
        continue
    if USE_WEIGHTED_MEAN:
        wts = g['quality_score'].fillna(0.5).to_numpy(dtype=float)
    else:
        wts = None
    tpl = make_template_from_indices(idxs, weights=wts)
    templates.append(tpl.astype(np.float32))
    rows.append({
        'person_id': person_id,
        'pose_cluster': int(pose_k),
        'n_images': int(len(idxs)),
        'mean_quality': float(g['quality_score'].mean(skipna=True)),
        'template_idx': tpl_idx,
        'type': 'hq_per_pose'
    })
    tpl_idx += 1

# Pooled fallback per-person (use fallback_mask so everyone gets covered)
for person_id, g in df[fallback_mask].groupby('person_id', sort=True):
    idxs = g.index.to_numpy().astype(int)
    if len(idxs) < 1:
        continue
    if USE_WEIGHTED_MEAN:
        wts = g['quality_score'].fillna(0.5).to_numpy(dtype=float)
    else:
        wts = None
    tpl = make_template_from_indices(idxs, weights=wts)
    templates.append(tpl.astype(np.float32))
    rows.append({
        'person_id': person_id,
        'pose_cluster': -1,
        'n_images': int(len(idxs)),
        'mean_quality': float(g['quality_score'].mean(skipna=True)),
        'template_idx': tpl_idx,
        'type': 'pooled_fallback'
    })
    tpl_idx += 1

if len(templates) == 0:
    raise RuntimeError("No templates created; check HQ/fallback masks and thresholds")

templates = np.vstack(templates).astype(np.float32)
templates_map = pd.DataFrame(rows)

# Save outputs
np.save(OUT_DIR / "templates_pose_hq.npy", templates)
templates_map.to_csv(OUT_DIR / "templates_map_pose_hq.csv", index=False)
print("Wrote HQ templates:", OUT_DIR / "templates_pose_hq.npy", "rows:", len(templates_map))

# Build FAISS index
templates /= np.linalg.norm(templates, axis=1, keepdims=True).clip(min=1e-12)
dim = templates.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(templates)
faiss.write_index(index, str(OUT_DIR / "templates_pose_hq.index"))
print("Wrote FAISS index:", OUT_DIR / "templates_pose_hq.index")

# Diagnostics
n_hq_per_pose = (templates_map['type'] == 'hq_per_pose').sum()
n_pooled = (templates_map['type'] == 'pooled_fallback').sum()
print(f"HQ per-pose templates: {n_hq_per_pose}; pooled fallback templates: {n_pooled}")
# persons missing any HQ per-pose template
persons_with_hq = templates_map[templates_map['type']=='hq_per_pose']['person_id'].unique()
all_persons = df['person_id'].unique()
missing_hq = set(all_persons) - set(persons_with_hq)
print("Persons missing HQ per-pose templates:", len(missing_hq))

Wrote HQ templates: ..\data_processed\vggface2\embeddings\enroll\templates_pose_hq.npy rows: 3218
Wrote FAISS index: ..\data_processed\vggface2\embeddings\enroll\templates_pose_hq.index
HQ per-pose templates: 2678; pooled fallback templates: 540
Persons missing HQ per-pose templates: 0


In [10]:
# check outputs
import numpy as np, pandas as pd
from pathlib import Path
base = Path("../data_processed/vggface2/embeddings/enroll")
print("templates_pose_hq:", (base / "templates_pose_hq.npy").exists())
print("map:", (base / "templates_map_pose_hq.csv").exists())
tpl = np.load(base / "templates_pose_hq.npy")
df = pd.read_csv(base / "templates_map_pose_hq.csv")
print("shape:", tpl.shape, "rows:", len(df))
print(df['type'].value_counts())
print(df.head())

templates_pose_hq: True
map: True
shape: (3218, 512) rows: 3218
type
hq_per_pose        2678
pooled_fallback     540
Name: count, dtype: int64
  person_id  pose_cluster  n_images  mean_quality  template_idx         type
0   n000001             0        15      0.704099             0  hq_per_pose
1   n000001             1        55      0.708452             1  hq_per_pose
2   n000001             2        86      0.715630             2  hq_per_pose
3   n000001             3        49      0.715965             3  hq_per_pose
4   n000001             4        16      0.719220             4  hq_per_pose
